In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import pandas as pd
import re
from bs4 import BeautifulSoup
import tokenizers
from torch.utils.data import DataLoader, Dataset
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore")

## Reading Data

In [2]:
text_docs = []
for i in range(16):
    with open(f"datasets/romegpt/rome_source_{i}", "r") as file:
        text_docs.append(file.read())

In [3]:
text_docs[0][:100]

'All Gaul is divided into three parts, one of which the Belgae inhabit, the Aquitani another, those w'

In [4]:
train_examples = []
for text_str in text_docs:
    current_substr = ""
    positions = [match.start() for match in re.finditer(r"\. ", text_str)]
    last_idx = 0
    for idx in positions:
        if(last_idx!=0):
            current_substr+=text_str[last_idx+1:idx]
        else:
            current_substr+=text_str[last_idx:idx]
        if(len(current_substr) > 20):
            if(len(current_substr) > 512):
                split_nums = len(current_substr)//512
                for split_idx in range(split_nums):
                    start_idx = split_idx*512
                    end_idx = (split_idx+1)*512
                    train_examples.append(current_substr[start_idx:end_idx].strip().lower())
            else:
                train_examples.append(current_substr.strip().lower())
            current_substr=""
        last_idx=idx

In [5]:
len(train_examples)

68406

In [6]:
# from datasets import load_dataset
# wiki = load_dataset("wikitext", "wikitext-103-v1", split="train")
# wiki = wiki.filter(lambda x: len(x["text"]) > 50)

## Tokenizer 

In [7]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel(add_prefix_space=True)
bpe_tokenizer.decoder = tokenizers.decoders.ByteLevel()
special_tokens = ["<unk>", "<pad>", "<s>", "</s>", "<mask>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=3000, special_tokens=special_tokens)
bpe_tokenizer.enable_padding(pad_id=1, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=100)
train_data = [t.lower() for t in text_docs]
bpe_tokenizer.train_from_iterator(train_data, bpe_trainer)

In [8]:
bpe_tokenizer.encode("Caesar crossed the Rubicon".lower()).ids

[1335, 1957, 138, 124, 2594, 159, 141]

## Model

In [9]:
from transformers import Trainer, TrainingArguments, PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from datasets import Dataset

device = "cuda"

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>",
    mask_token="<mask>"
)

rome_dataset = Dataset.from_dict({"text": [t.lower() for t in train_examples]})

def tokenize(example):
    return wrapped_tokenizer(["<s>" + t + "</s>" for t in example["text"]], truncation=True, max_length=512, padding="max_length")

tokenized = rome_dataset.map(tokenize, batched=True, remove_columns=["text"])
# wiki_tokenized = wiki.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(wrapped_tokenizer, mlm=False)

Map: 100%|██████████| 68406/68406 [00:03<00:00, 19333.50 examples/s]


In [10]:
class RomeLM(nn.Module):
    def __init__(self, vocab_size, d_model=1024, nhead=8, num_layers=10, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Embedding(512, d_model)  # learned positional
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.pad_token_id = wrapped_tokenizer.pad_token_id
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_encoding(positions)
        
        # causal mask so each token only attends to previous tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=key_padding_mask)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            shift_labels[shift_labels == self.pad_token_id] = -100

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )

        from transformers.modeling_outputs import CausalLMOutput
        return CausalLMOutput(loss=loss, logits=logits)

In [11]:
romeGPT = RomeLM(wrapped_tokenizer.vocab_size, d_model=512, nhead=8, num_layers=6, dropout=0.1).to(device)
args = TrainingArguments(output_dir="./rome_gpt", num_train_epochs=3, per_device_train_batch_size=64, remove_unused_columns=False, logging_steps=200)
trainer = Trainer(model=romeGPT, args=args, train_dataset=tokenized, data_collator=collator)
trainer_output = trainer.train()

Step,Training Loss
200,6.402949
400,5.850322
600,5.583384
800,5.419018
1000,5.291978
1200,5.173614
1400,5.084872
1600,5.019473
1800,4.951901
2000,4.914063


In [12]:
# args = TrainingArguments(output_dir="./rome_gpt", num_train_epochs=4, per_device_train_batch_size=64, remove_unused_columns=False, logging_steps=500)
# trainer = Trainer(model=romeGPT, args=args, train_dataset=tokenized, data_collator=collator)
# trainer.train(resume_from_checkpoint=True)

In [13]:
def generate(model, tokenizer, prompt, max_new_tokens=20, temperature=0.8):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [19]:
generate(romeGPT.to(device), wrapped_tokenizer, "Caesar crossed the Rubicon and".lower(), temperature=0.9)

' caesar crossed the rubicon and the diffe writersber, on the senate, which had letaled to the cavalry, who were able'

In [15]:
torch.save(romeGPT.state_dict(), "rome_gpt.pt")

In [16]:
dummy_input = torch.zeros(1, 100, dtype=torch.long).cpu()

torch.onnx.export(
    romeGPT.eval().cpu(),
    (dummy_input,),
    "rome_gpt.onnx",
    input_names=["input_ids"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq_len"},
        "logits": {0: "batch", 1: "seq_len"},
    },
    opset_version=17,
    dynamo=False,
)

In [17]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic("rome_gpt.onnx", "rome_gpt_quantized.onnx", weight_type=QuantType.QInt8)

In [20]:
import json

config = {
    "tokenizer_class": "PreTrainedTokenizerFast",
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "mask_token": "<mask>",
    "model_max_length": 100,
}

with open("tokenizer_config.json", "w") as f:
    json.dump(config, f)

In [21]:
bpe_tokenizer.save("tokenizer.json")